---
title: "Data Cleaning"
format:
    html: 
        code-fold: false
---

<!-- After digesting the instructions, you can delete this cell, these are assignment instructions and do not need to be included in your final submission.  -->

{{< include instructions.qmd >}} 

First, I read the original CSV data files that had been crawled. However, the original file contained a lot of redundant information and formatting issues, such as multiple header rows, comment symbols, blank characters, as well as incorrect or duplicate column names. To deal with these problems, I renamed and adjusted the column names, cleaned the square bracket comments and spaces at both ends of the text data, as well as the incorrect or duplicate column names. In this process, I renamed and adjusted the column names, cleaned the square bracket comments and spaces in the text data, and replaced the special character "—" with the missing value NaN. 

Then, I converted all the columns related to appearances and goals into numeric types to prepare for subsequent calculations, as well as for supervised and unsupervised learning. I filtered out the summary rows containing "Total", and reset the index to maintain data continuity.

Finally, I saved the cleaned club and national team data as new CSV files for convenient use in subsequent analyses.

In [9]:
import pandas as pd
import numpy as np

In [10]:
df1 = pd.read_csv('../../data/raw-data/messi_club_career.csv', header=[0,1])

# Rename columns for easier access
df1.columns = [
    'Club', 'Season',
    'League_Apps', 'League_Goals', 'League_Apps_2',
    'Cup_Apps', 'Cup_Goals',
    'Continental_Apps', 'Continental_Goals',
    'Other_Apps', 'Other_Goals',
    'Total_Apps', 'Total_Goals'
]

# Swap mistaken League_Apps and League_Goals columns
df1['League_Apps'], df1['League_Goals'] = df1['League_Goals'], df1['League_Apps_2']
df1.drop(columns=['League_Apps_2'], inplace=True)

# Remove footnote marks like [1], [2]
df1 = df1.replace(r'\[.*?\]', '', regex=True)

# Strip whitespace
df1 = df1.applymap(lambda x: x.strip() if isinstance(x, str) else x)

# Replace em-dash with NaN
df1.replace('—', np.nan, inplace=True)


# Convert numeric columns to proper numeric types
num_cols = [col for col in df1.columns if 'Apps' in col or 'Goals' in col]
for col in num_cols:
    df1[col] = pd.to_numeric(df1[col], errors='coerce')

# Remove total summary rows
df1 = df1[~df1['Season'].str.contains('Total', na=False)]

# Reset index
df1.reset_index(drop=True, inplace=True)




#  International Career data use the same methods

df2 = pd.read_csv('../../data/raw-data/messi_international_career.csv', skiprows=1)
df2 = df2.replace(r'\[.*?\]', '', regex=True)
df2 = df2.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df2.replace('—', np.nan, inplace=True)
df2.columns = [
    'Team', 'Year',
    'Competitive_Apps', 'Competitive_Goals',
    'Friendly_Apps', 'Friendly_Goals',
    'Total_Apps', 'Total_Goals'
]


num_cols = [col for col in df2.columns if '_Apps' in col or '_Goals' in col]
for col in num_cols:
    df2[col] = pd.to_numeric(df2[col], errors='coerce')


df2 = df2[~df2['Year'].str.contains('Total', na=False)]


df2.reset_index(drop=True, inplace=True)


df1.to_csv('../../data/processed-data/messi_club_career_clean.csv', index=False, encoding='utf-8-sig')
df2.to_csv('../../data/processed-data/messi_national_career_clean.csv', index=False, encoding='utf-8-sig')

/var/folders/fd/zfxd46k90qd0bd_xk04w45r40000gn/T/ipykernel_86705/3237841233.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df1 = df1.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/fd/zfxd46k90qd0bd_xk04w45r40000gn/T/ipykernel_86705/3237841233.py:45: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df2 = df2.applymap(lambda x: x.strip() if isinstance(x, str) else x)


Now i read the detailed match data of Messi. It will serve as the basic dataset for subsequent supervised learning and unsupervised learning.

In [11]:
df3 = pd.read_csv('../../data/raw-data/messi_detailed.csv')
# Print the number of columns
print("Columns after cleaning:", len(df3.columns))


Columns after cleaning: 275


Subsequently, I conducted column filtering on the data, retaining only the key columns such as player basic information, game statistics indicators, discipline indicators, as well as league and club goal-scoring rankings, and deleting redundant or irrelevant columns.

In [12]:
# the cloumns we need
columns = [
    "full_name",
    "league",
    "season",
    "Current Club",
    "minutes_played_overall",
    "appearances_overall",
    "goals_overall",
    "assists_overall",
    "penalty_goals",
    "penalty_misses",  
    "goals_involved_per_90_overall",
    "assists_per_90_overall",
    "goals_per_90_overall",
    "yellow_cards_overall",
    "red_cards_overall",
    "min_per_card_overall",   
    "rank_in_league_top_attackers",
    "rank_in_league_top_midfielders",
    "rank_in_league_top_defenders",
    "rank_in_club_top_scorer"
]

df3 = df3[columns]

print("Columns after selection:", len(df3.columns))
print(df3.columns.tolist())


Columns after selection: 20
['full_name', 'league', 'season', 'Current Club', 'minutes_played_overall', 'appearances_overall', 'goals_overall', 'assists_overall', 'penalty_goals', 'penalty_misses', 'goals_involved_per_90_overall', 'assists_per_90_overall', 'goals_per_90_overall', 'yellow_cards_overall', 'red_cards_overall', 'min_per_card_overall', 'rank_in_league_top_attackers', 'rank_in_league_top_midfielders', 'rank_in_league_top_defenders', 'rank_in_club_top_scorer']


The format of the season column in the original data was very messy, including "2012-2013", "2018" or "2018/2019". Therefore, I standardized the season column by converting all the different formats into the "start/end" form and extracted the year of season end, so as to sort by time, group for analysis and conduct machine learning modeling in the future.

In [13]:
# Standardize the 'season' column in df3:
# - Converts all season entries to a consistent "start_year/end_year" format
# - Creates a new 'end' column representing the ending year of the season
# This helps with sorting, filtering, and analysis of seasonal data.
def unify_season(season):
    season = str(season).strip()
    if "/" in season:
        start, end = season.split("/")
    else:
        start = season
        end = str(int(start))
    unified = f"{start}/{end}"
    return unified, end  
df3[['season', 'end']] = df3['season'].apply(lambda x: pd.Series(unify_season(x)))


I applied hierarchical cleaning and standardization for the four ranking indicators for the players (rank_in_league_top_attackers, rank_in_league_top_midfielders, rank_in_league_top_defenders, rank_in_club_top_scorer). The ranking values in the original data vary significantly across different dimensions, and some players are marked as -1 in dimensions that are not applicable (for example, defenders have no ranking for attackers). To facilitate model training and feature comparison, we converted all ranking values to a scale of 0–5 (Level), where a higher Level indicates better performance, and -1 is defined as Level 0 (invalid or unranked). The level classification is based on the percentile intervals of each column: the top 5% is set as Level 5, 5–20% as Level 4, 20–50% as Level 3, 50–80% as Level 2, and 80% and above as Level 1.

In [14]:
cols = [
    "rank_in_league_top_attackers",
    "rank_in_league_top_midfielders",
    "rank_in_league_top_defenders",
    "rank_in_club_top_scorer"
]

def rank_to_level(series): # Function to convert numeric ranks into level scores (1-5)
    s = series.copy()
    s = s.replace(-1, np.nan)  
    quantiles = s.quantile([0.05, 0.2, 0.5, 0.8]).to_dict()

    def map_func(x):
        if pd.isna(x): return 0
        if x <= quantiles[0.05]: return 5
        elif x <= quantiles[0.2]: return 4
        elif x <= quantiles[0.5]: return 3
        elif x <= quantiles[0.8]: return 2
        else: return 1
    return series.apply(map_func)

for col in cols:
    df3[col] = rank_to_level(df3[col])


Finally, i combined the columns related to penalties into a single boolean feature named "has_penalty", indicating whether the player has ever taken a penalty kick: if the number of penalty goals or penalty misses is greater than 0, the value of this column is True; otherwise, it is False. This simplifies the data while retaining the key information.

In [15]:
penalty_cols = ["penalty_goals", "penalty_misses"]
# Create a new column indicating whether the player took any penalties in the match
df3["has_penalty"] = (df3["penalty_goals"] + df3["penalty_misses"]) > 0

# Drop the original penalty columns as they are no longer needed
df3.drop(columns=penalty_cols, inplace=True)

Save the cleaned data as new CSV files for subsequent analysis and modeling.

In [16]:
#save
df3.to_csv('../../data/processed-data/messi_club_detailed_cleaned.csv', index=False, encoding='utf-8-sig')